# CS3120/5120: Secure Distributed Computation
## In-Class Exercise, Week of 2/9/2026

In [ ]:
import pychor
import galois
import numpy as np

p = (2**31)-1
GF = galois.GF(p)
p1 = pychor.Party('p1')
p2 = pychor.Party('p2')

@pychor.local_function
def share(x):
    s1 = GF.Random()
    s2 = GF(x) - s1
    return s1, s2

## Question 1

Why is the OT protocol secure against a semi-honest adversary?

YOUR ANSWER HERE

## Question 2

Why is the protocol not secure against a malicious adversary?

YOUR ANSWER HERE

## Question 3

Why should we care about OT?

YOUR ANSWER HERE

## Question 4

Describe a strategy for generating a multiplication triple $a$, $b$, $c$ in $GF(2)$ (binary) such that $ab = c$.

1. The parties generate shares of $a$ and $b$ randomly
2. P1 generates a share of $c$ randomly
3. P1 and P2 run 1-out-of-4 OT with P1 as sender and P2 as receiver to deliver P2's share of $c$ to P2. The inputs are the 4 possible values for P2's share of $c$, and the selection index is built from P2's shares of $a$ and $b$

## Question 5

Write a function to generate the possible values for P2's share of $c$.

In [ ]:
GF_2 = galois.GF(2)

def truth_table(a1, b1, c1):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
assert truth_table(GF_2(0), GF_2(0), GF_2(0)) == [0, 0, 0, 1]

## Question 6

Implement a protocol for generating a binary multiplication triple. Reference [Chapter 6 of the textbook](https://jnear.github.io/programming-mpc/chapters/chapter06.html#application-generating-binary-multiplication-triples-using-ot).

In [ ]:
def protocol_gen_binary_mult_triple():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# Protocol for checking a triple by reconstruction
def test_triple(triple):
    (a1, a2), (b1, b2), (c1, c2) = triple
    a2.send(p2, p1)
    b2.send(p2, p1)
    c2.send(p2, p1)
    ab = (a1+a2) * (b1+b2)
    c = c1+c2
    print(f'a*b: {ab}, c: {c}; are they equal? {ab == c}')

with pychor.LocalBackend():
    for _ in range(5):
        # Generate a triple
        triple = protocol_gen_binary_mult_triple()
        # Verify it's correct
        test_triple(triple)

## Question 7

Multiply 3 and 12 by *bit-decomposing* 12.

In [ ]:
def bit_decompose(x, nbits):
    # Convert the input to an integer
    x = int(x)
    # Return a list of bits corresponding to x
    # the least-significant bit is the first element
    return [(x >> i) & 1 for i in range(nbits)]

In [ ]:
def mult_3_12():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
ab = mult_3_12()
print('ab:', ab)
assert ab == 3*12

## Question 8

Describe a protocol for computing secret shares of a product $ab$ where P1 knows $a$ and P2 knows $b$. In other words, one input is fully known to each of the parties, and the parties should each receive one secret share of the product.

YOUR ANSWER HERE

## Question 9

Implement the protocol for computing secret shares of $ab$.

In [ ]:
def protocol_crossterm(a1, b2):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    a1 = p1.constant(GF(1))
    b2 = p2.constant(GF(4))
    print('Cross term shares:', protocol_crossterm(a1, b2))

## Question 10

Describe the protocol for generating an arithmetic multiplication triple.

YOUR ANSWER HERE

## Question 11

Implement a protocol for generating an arithmetic multiplication triple.

In [ ]:
def protocol_gen_arithmetic_mult_triple():
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    print('Multiplication triple:', protocol_gen_arithmetic_mult_triple())

## Question 12

Describe the GMW protocol for MPC. Reference [Chapter 7 of the textbook](https://jnear.github.io/programming-mpc/chapters/chapter07.html).

YOUR ANSWER HERE

## Question 13

Implement the GMW protocol as a class called `SecBit`. Reference (copy/paste from) the textbook.

In [ ]:
from dataclasses import dataclass

def protocol_gmw_mult(x, y):
    # YOUR CODE HERE
    raise NotImplementedError()
    
@pychor.local_function
def share(x):
    s1 = GF_2.Random()
    s2 = GF_2(x) - s1
    return s1, s2

@dataclass
class SecBit:
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    x_input = p1.constant(1)
    y_input = p2.constant(0)

    # Create secret shares of the inputs
    x = SecBit.input(x_input)
    y = SecBit.input(y_input)

    # Online phase: compute (x+y)^3
    z = x + y
    result = z*z*z
    print('(x+y)^3:', result.reveal())

## Question 14

Describe how to extend the GMW approach to $n$ parties. Reference [slide 34 of these slides from UIUC CS 598](https://courses.grainger.illinois.edu/cs598man/sp2016/slides/17.pdf).

YOUR ANSWER HERE

## Question 15

Describe how to extend the GMW approach to arithmetic secret shares (in GF(p)).

YOUR ANSWER HERE

# OT Protocol (for use above)

In [ ]:
from nacl.public import PrivateKey, PublicKey, SealedBox
from nacl.utils import random

def protocol_ot(sender, receiver, inputs, selection, n):
    # Function for the Receiver to generate keys
    @pychor.local_function
    def gen_keys(selection, n):
        # Generate a single real key pair key = (sk, pk)
        key = PrivateKey.generate()
        public_keys = [PublicKey(random(PublicKey.SIZE)) for _ in range(n)]
        public_keys[selection] = key.public_key
        return key, public_keys

    # Function for the Sender to encrypt the secret inputs
    @pychor.local_function
    def encrypt_inputs(pub_keys, inputs):
        # Encode the inputs as bytes
        length = max([(int(x).bit_length() + 7) // 8 for x in inputs])
        inputs_bytes = [int(x).to_bytes(length, 'little') for x in inputs]
    
        # Encrypt the inputs
        encrypted_inputs = [SealedBox(pk).encrypt(x) for pk, x in \
                            zip(pub_keys, inputs_bytes)]
        return encrypted_inputs

    # Function for the Receiver to decrypt the result
    @pychor.local_function
    def decrypt_result(selection, key, encrypted_inputs):
        # Select the correct input
        selected_input = encrypted_inputs[selection]
        # Decrypt it and convert it from bytes to int
        plaintext = SealedBox(key).decrypt(selected_input)
        return int.from_bytes(plaintext, 'little')

    # Step 1: Generate keys and send to Sender
    sk, pub_keys = gen_keys(selection, n).untup(2)
    pub_keys.send(receiver, sender)

    # Step 2: Encrypt inputs and send to Receiver
    encrypted_inputs = encrypt_inputs(pub_keys, inputs)
    encrypted_inputs.send(sender, receiver)

    # Step 3: Decrypt result
    result = decrypt_result(selection, sk, encrypted_inputs)

    return result